# Adapting Gemma 4 to Brazilian Portuguese — Colab GPU Pipeline

Este notebook executa o pipeline completo — **treino (CPT piloto em QLoRA) → recuperação de instruction-following (residual merge) → avaliação com métricas e intervalos de confiança** — numa única GPU do Google Colab.

**O que este notebook faz:**
1. Instala as dependências certas (sem reinstalar o PyTorch/CUDA que já vem pronto no Colab).
2. Clona (ou usa) este repositório.
3. Roda um piloto de Continued Pre-Training (CPT) do **Gemma 4 E2B** (o menor modelo real da família Gemma 4 — Apache 2.0, não-gated) em QLoRA (4-bit) no corpus **Aurora-PT**, com replay de inglês para mitigar esquecimento catastrófico.
4. Recupera a capacidade de seguir instruções via **residual merge** (aritmética de tensores, sem treino adicional — Ilharco et al. 2023 / "chat vector", Huang et al. 2024).
5. Avalia o modelo resultante num subconjunto de benchmarks reais em português (ENEM, BLUEX, ASSIN2, HateBR, OAB) + retenção em inglês (MMLU), com **bootstrap/Wilson confidence intervals** — não apenas um número solto.
6. Gera um dashboard de resultados e mostra tudo inline.

**GPU recomendada:** Runtime → Change runtime type → GPU. Funciona em T4 (grátis, 16GB — ajuste `per_device_train_batch_size` para baixo se faltar VRAM), L4 (24GB) ou A100 (40GB, Colab Pro/Pro+) — todas suportadas pela config QLoRA usada aqui.

**Tempo esperado:** o piloto (300 steps, seq_len=2048, batch efetivo ~65K tokens) leva de alguns minutos (A100) a ~30-60min (T4) — pensado para caber com folga numa sessão. Para uma run "de verdade" (mais tokens), aumente `max_steps` em `configs/train/cpt_colab_pilot.yaml` e use os checkpoints no HF Hub para retomar entre sessões (ver seção de persistência abaixo).

**Checkpoints não sobrevivem ao fim da sessão do Colab por padrão.** Este notebook empurra checkpoints para o seu HF Hub (`push_to_hub`) — configure seu `HF_TOKEN` nos Colab Secrets (ícone de chave 🔑 na barra lateral) antes de rodar.

## 0. Verificar GPU disponível

In [ ]:
!nvidia-smi

## 1. Clonar o repositório

Se você já editou/fez fork do repo, troque a URL abaixo. Se está rodando a partir de um checkout já presente no ambiente (ex.: Colab conectado a um repo local via Drive), pule esta célula e apenas ajuste o `%cd`.

In [ ]:
REPO_URL = "https://github.com/vfcarida/Adapting-Gemma-4-to-Brazilian-Portuguese.git"
REPO_DIR = "Adapting-Gemma-4-to-Brazilian-Portuguese"

import os
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR

## 2. Instalar dependências

Instalamos **apenas** os pacotes de cima do PyTorch/CUDA que o Colab já traz pré-configurado e testado — reinstalar `torch` costuma quebrar a build CUDA do ambiente. As versões abaixo espelham `pyproject.toml` (Gemma 4 requer `transformers>=5.5.0`).

In [ ]:
!pip install -q -U \
  "transformers>=5.5.0" "trl>=1.0.0" "peft>=0.14.0" "accelerate>=1.2.0" \
  "datasets>=3.0.0,<4.0.0" "bitsandbytes>=0.45.0" \
  sentencepiece protobuf scipy scikit-learn pandas pyyaml python-dotenv \
  rich typer tqdm jsonlines xxhash "datasketch>=2.0.0" nltk tabulate

# Instala o pacote deste repo em modo editável, sem reprocessar dependências
# (já instaladas acima) para não arriscar um upgrade/downgrade do torch.
!pip install -q -e . --no-deps

## 3. Autenticação (HF Hub e, opcionalmente, W&B)

Configure `HF_TOKEN` (obrigatório para `push_to_hub`; os modelos Gemma 4 e o dataset Aurora-PT usados aqui são públicos/não-gated, então o token NÃO é necessário só para baixar) e, opcionalmente, `WANDB_API_KEY` nos **Colab Secrets** (ícone de chave 🔑 na barra lateral esquerda) antes de rodar esta célula.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("HF Hub: autenticado.")
except Exception as e:
    print(f"HF Hub: sem token configurado ({e}). Download de modelos públicos ainda funciona; "
          "push_to_hub para checkpoints NÃO vai funcionar sem um token com permissão de escrita.")

import os
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B: chave carregada (opcional — troque report_to para incluir 'wandb' na config se quiser usá-lo).")
except Exception:
    print("W&B: não configurado (opcional, ok pular).")

## 4. Preflight: validar ambiente

Confere GPU/VRAM, versões de pacotes, autenticação HF e configs — antes de gastar tempo de sessão com algo que vai falhar de qualquer forma.

In [ ]:
!python -m src.cli preflight

## 5. (Opcional) Smoke test rápido da config de treino

Roda apenas 10 steps (batch_size=1, forçado por --tiny) para confirmar que o pipeline de dados/modelo/treino funciona de ponta a ponta antes de comprometer a sessão inteira ao treino completo.

In [ ]:
!python -m src.cli train-cpt configs/train/cpt_colab_pilot.yaml --tiny

## 6. Continued Pre-Training (CPT) — piloto QLoRA no Gemma 4 E2B

Usa `configs/train/cpt_colab_pilot.yaml`: Gemma 4 E2B, QLoRA (r=64, α=128, todos os módulos lineares), `learning_rate=2e-5` (faixa recomendada pela literatura para CPT com LoRA — não a regra "10x maior" que vale para instruction tuning), replay de 15% inglês (FineWeb-Edu) misturado ao Aurora-PT, `max_steps=300`.

**Para uma run maior:** edite `training.max_steps` no YAML (ou use `--override training.max_steps=3000`), e considere ativar `output.push_to_hub: true` + `output.hub_model_id` no YAML para poder retomar em outra sessão (checkpoint automático a cada `save_steps`, e resume automático via `find_latest_checkpoint`/`--resume`).

In [ ]:
!python -m src.cli train-cpt configs/train/cpt_colab_pilot.yaml

## 7. Recuperar instruction-following: Residual Merge

Sem treino adicional: `merged = CPT + alpha * (instruct - base)` (task arithmetic, Ilharco et al. 2023). Testamos alpha=1.0 (transferência completa do vetor de instrução) — ajuste a lista de `--alpha` para fazer um sweep se tiver tempo de sessão sobrando (ex.: `--alpha 0.6 0.8 1.0`).

In [ ]:
!python -m src.train.residual_merge \
    --base-model google/gemma-4-E2B \
    --instruct-model google/gemma-4-E2B-it \
    --cpt-model outputs/cpt_colab_pilot/final \
    --alpha 1.0 \
    --output-dir outputs/residual_merge

## 8. Avaliação: benchmarks reais em português + retenção em inglês

Usa `configs/eval/benchmarks_colab.yaml`: um subconjunto de ~6 benchmarks reais e verificados (ENEM, BLUEX, ASSIN2-RTE, HateBR, OAB, MMLU-EN) escolhido para caber numa sessão — não a suite completa de 20+ benchmarks de `configs/eval/benchmarks.yaml` (essa é a opção para uma run "de verdade" fora do Colab, ou numa sessão dedicada de avaliação). Usa logprob scoring (mais rápido que gerar texto completo) para as tarefas de múltipla escolha, e calcula Wilson score intervals / bootstrap CI por benchmark — não só um ponto.

In [ ]:
!python -m src.cli eval --config configs/eval/benchmarks_colab.yaml

## 9. Relatório e dashboard

In [ ]:
!python -m src.cli report
!python scripts/build_dashboard.py --format markdown

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

df = pd.read_csv("reports/results_full.csv")
display(df)

with open("reports/dashboard.md") as f:
    display(Markdown(f.read()))

## 10. Persistir resultados (o Colab NÃO mantém disco entre sessões)

Duas opções, use uma ou ambas:
- **HF Hub** (recomendado — já usado para checkpoints se você ativou `push_to_hub`): suba os relatórios também, para um repo de dataset.
- **Google Drive**: monte o Drive e copie `outputs/` e `reports/` para lá.

In [ ]:
# Opção A — subir reports/ para um repo de dataset no HF Hub
from huggingface_hub import HfApi

REPORTS_REPO = None  # ex.: "seu-usuario/gemma4-ptbr-colab-reports"
if REPORTS_REPO:
    api = HfApi()
    api.create_repo(REPORTS_REPO, repo_type="dataset", exist_ok=True, private=True)
    api.upload_folder(repo_id=REPORTS_REPO, repo_type="dataset", folder_path="reports", path_in_repo="reports")
    print(f"Reports enviados para https://huggingface.co/datasets/{REPORTS_REPO}")
else:
    print("Defina REPORTS_REPO acima para habilitar o upload.")

In [ ]:
# Opção B — copiar para o Google Drive
MOUNT_DRIVE = False  # troque para True para habilitar
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/gemma4-ptbr-colab
    !cp -r outputs reports /content/drive/MyDrive/gemma4-ptbr-colab/
    print("Copiado para /content/drive/MyDrive/gemma4-ptbr-colab/")

## Próximos passos

- **Escalar o treino**: aumente `training.max_steps` (e o orçamento de tokens) em `configs/train/cpt_colab_pilot.yaml`, ou passe para `configs/train/cpt_pilot.yaml`/`cpt_main.yaml` (Gemma 4 E4B/26B, full fine-tune, múltiplas GPUs — ver `docs/TRAINING_GUIDE.md` e `infra/gcp/` para a trilha GCP multi-GPU).
- **Avaliação completa**: rode `configs/eval/benchmarks.yaml` (20+ benchmarks, think mode on/off, todos os baselines) numa sessão dedicada — é bem mais demorado que o subconjunto usado aqui.
- **SFT em vez de (ou além do) residual merge**: `python -m src.cli train-sft configs/train/sft.yaml` usando `outputs/cpt_colab_pilot/final` como `base_checkpoint`.
- **Retomar treino interrompido**: se você ativou `push_to_hub` com `hub_strategy: "checkpoint"`, baixe `last-checkpoint/` do seu repo HF (`snapshot_download`) para `outputs/cpt_colab_pilot/` numa nova sessão e rode o mesmo comando de treino — `find_latest_checkpoint` retoma automaticamente.
- Ver `docs/CPT_BEST_PRACTICES_RESEARCH.md` para a literatura por trás de cada escolha (replay ratio, LR, rank do LoRA, alpha do merge).